In [1]:
import importlib
import parser as parser_module
importlib.reload(parser_module)

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from datetime import date

from parser import parse_factsheet, format_for_llm, extract_key_fields, extract_factsheet_date
from scraper import select_relevant_links, download_pdf, fetch_page_text
from enricher import enrich, format_enrichment_for_llm

load_dotenv(override=True)
openai = OpenAI()
MODEL = "gpt-4o"

In [2]:
dd_system_prompt = """
You are a senior investment analyst producing a due diligence brief for an institutional investor.
You will be given extracted text from a fund factsheet and supporting market data from yFinance.
Produce a structured brief in markdown. Be precise, use numbers where available, and avoid filler language.

CRITICAL RULES:
- Only use facts explicitly stated in the provided text. Do not infer or hallucinate any data.
- If a field is missing or unclear, write "Not disclosed" rather than guessing.
- Fund domicile and legal structure are NOT the same as portfolio geographic allocation.
- The benchmark listed in the factsheet may be "none" — state this honestly if so.
- The ## Market Context section contains benchmark proxy data from yFinance — use this 
  explicitly in the Performance section to compare fund returns against the proxy.
- For VaR, always include the confidence level, time horizon and exact percentage figure.
- When a fund has a stated absolute return target (e.g. "money market +2.5%"), 
  evaluate performance against that target first, not just vs the proxy ETF.
- VaR figures: ALWAYS use values from the ## Key Risk Figures (extracted) section, 
  never from raw text. The format "VaR 95 -10" means 10-day horizon, not the value.

Structure your output exactly as follows:

## Fund at a Glance
One paragraph: fund name, manager(s), AUM, domicile, inception date, stated benchmark 
(write "No benchmark" if none listed), SFDR classification.

## Investment Strategy
How the manager selects securities. Investment universe size, long/short approach if applicable,
key differentiators. Use exact quotes from the factsheet where helpful.

## Portfolio Characteristics
Investment style, actual portfolio geographic allocation (from factsheet charts/tables, 
NOT fund domicile), sector allocation with percentages, net equity exposure if stated.

## Performance
Fund returns across all available periods (YTD, 1Y, 3Y, 5Y, since inception).
Compare explicitly against the benchmark proxy from the Market Context section,
labelling it clearly as a proxy. If the fund has a stated return target, evaluate
performance against that target explicitly. Note max drawdown and return consistency across years.

## Risk Profile
Volatility p.a., Sharpe ratio, max drawdown, VaR 95 and VaR 99 with exact figures,
correlation to benchmark. Risk indicator rating (1-7 scale).

## Costs
TER (with date), management fee, performance fee with exact hurdle rate and 
high-watermark details. Entry/exit fees if any.

## Analyst Verdict
3-5 sentences. What type of investor or mandate this fund suits. Key strengths and weaknesses.
This brief is a first-pass screening tool designed to triage funds for deeper human review.
Make a decisive call based on available data:
- "Suitable" — fund has clear fit for institutional mandates, consistent track record, 
  reasonable costs, and meets or exceeds its stated return target
- "Requires Further Due Diligence" — genuinely ambiguous cases only: insufficient data, 
  unusual fee structure, inconsistent performance, or significant unexplained risks
- "Not Suitable" — clear misfit: excessive costs, poor risk-adjusted returns, or structural concerns
- A fund consistently meeting its stated return target with low costs, no performance fee, 
  and institutional-grade structure should be rated Suitable unless there is a specific 
  identified concern. Outperformance over 3Y vs proxy is a strong positive signal.

Recommendation: Suitable / Requires Further Due Diligence / Not Suitable — one-line rationale.
"""

In [3]:
def build_brief(fund_name: str, url: str, benchmark_hint: str = "", fund_ticker: str = ""):
    
    # step 1: scrape links
    print(f"Step 1: Scraping {url}...")
    docs = select_relevant_links(url)
    
    # step 2: find and download factsheet
    print("Step 2: Downloading factsheet...")
    factsheet_path = None
    for doc in docs.get("documents", []):
        if doc["type"] == "factsheet":
            factsheet_path = f"briefs/{fund_name.replace(' ', '_')}_factsheet.pdf"
            success = download_pdf(doc["url"], factsheet_path)
            if not success:
                factsheet_path = None
            break
    
    # step 3: parse factsheet
    print("Step 3: Parsing factsheet...")
    if factsheet_path:
        sections = parse_factsheet(factsheet_path)
        factsheet_text = format_for_llm(sections)
        factsheet_date = extract_factsheet_date(sections)
        print(f"  Factsheet date detected: {factsheet_date}")
    else:
        print("  No factsheet found, falling back to page text...")
        factsheet_text = fetch_page_text(url)
        factsheet_date = None

    # step 4: enrich with yfinance
    print("Step 4: Fetching market data...")
    enrichment = enrich(
        benchmark_hint=benchmark_hint,
        fund_ticker=fund_ticker,
        as_of_date=factsheet_date
    )
    enrichment_text = format_enrichment_for_llm(enrichment)
    
    # step 5: assemble prompt
    today = date.today().strftime("%d %B %Y")
    user_prompt = f"""
Fund Name: {fund_name}
Brief Date: {today}

{factsheet_text[:8000]}

{enrichment_text}
"""
    
    # step 6: stream the brief
    print("Step 5: Generating DD brief...\n")
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": dd_system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        stream=True
    )

    full_response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        full_response += chunk.choices[0].delta.content or ""
        update_display(Markdown(full_response), display_id=display_handle.display_id)

    # step 7: save to file
    output_path = f"briefs/{fund_name.replace(' ', '_')}_{date.today().strftime('%Y%m%d')}.md"
    with open(output_path, "w") as f:
        f.write(f"# {fund_name} — Due Diligence Brief\n")
        f.write(f"*Generated: {today}*\n\n")
        f.write(full_response)
    print(f"\nBrief saved to {output_path}")

In [6]:
build_brief(
    fund_name="Lupus alpha CLO High Quality Invest",
    url="https://www.lupusalpha.com/products/fund/lupus-alpha-clo-high-quality-invest-a/",
    benchmark_hint="clo"
)

Step 1: Scraping https://www.lupusalpha.com/products/fund/lupus-alpha-clo-high-quality-invest-a/...
Found 75 total links, filtering with LLM...
Step 2: Downloading factsheet...
Downloaded: briefs/Lupus_alpha_CLO_High_Quality_Invest_factsheet.pdf
Step 3: Parsing factsheet...
  Factsheet date detected: 29.05.2026
Step 4: Fetching market data...
Fetching benchmark data for: IS0R.DE anchored to 29.05.2026
Step 5: Generating DD brief...



## Fund at a Glance
Lupus alpha CLO High Quality Invest, managed by Stamatia Michael, Norbert Adam, and Dr. Klaus Ripper, has assets under management of EUR 217.46 million. The fund is domiciled in Germany and was incepted on July 1, 2015. It has no stated benchmark and is classified under SFDR Article 8.

## Investment Strategy
The fund invests in a diversified portfolio of secured corporate loans through collateralized loan obligations (CLOs), restricted to securities with a minimum investment grade rating of BBB-/Baa3. The strategy integrates ESG criteria and aims for a return of money market +2.5% annually, primarily derived from coupon payments. The fund targets low single-digit volatility.

## Portfolio Characteristics
The investment style focuses on loans and CLOs with investment grade credit quality. The geographical allocation is not disclosed.

## Performance
- Year to Date: 1.66%
- 1 Year: 4.26%
- 3 Years: 22.03%
- 5 Years: 21.27%
- Since Inception (annualized): 2.73%

Comparatively, the benchmark proxy (IS0R.DE) shows a 1-year performance of 4.46% and a 3-year performance of 17.83%.

The fund has a maximum drawdown of -15.62%.

## Risk Profile
- Volatility p.a.: 3.64%
- Sharpe Ratio: 0.57
- Max Drawdown: -15.62%
- VaR 95: Not disclosed
- VaR 99: Not disclosed
- Correlation to benchmark: Not disclosed
- Risk Indicator Rating: 2 (on a scale of 1-7)

## Costs
- Total Expense Ratio (TER): 0.71% p.a. as of 30.11.2025
- Management Fee: 0.60%
- Performance Fee: None
- Initial Charge: Up to 4%

## Analyst Verdict
The Lupus alpha CLO High Quality Invest fund is suitable for institutional investors seeking exposure to investment-grade corporate loans with an ESG overlay. The fund provides potential for attractive credit spreads with low volatility and has consistently met its stated return target of money market +2.5% annually. While the fund’s performance fees are low and it has a strong track record, the high initial charge could be a deterrent. Nonetheless, it outperformed the benchmark proxy over 3 years, making it a strong candidate for institutional mandates.

Recommendation: Suitable — consistent performance against targets and low cost structure.


Brief saved to briefs/Lupus_alpha_CLO_High_Quality_Invest_20260604.md
